# Sentiment Analysis - VADER
* VADER (Valence Aware Dictionary and sEntimnet Reasoner): Is a lexicon and rule-based model for sentiment analysis. 
* Sentiment Analysis:
    * polarity-based: pieces of texts are classified as either positive or negative
    * valence-based: intensity of the sentiment is taken into account. 
* VADER:
    * Each word in the lexico has a sentiment rating
    * The text is analyzed with four ratings: positive, negative, neutral and compound. 
        * Positive, Negative and Neutral represent the proportion of the text that falls into these categories. They are on the scale from 0 to 1. 
        * Compound is the sum of valence scores of each word in the lexicon determines the degeree of the sentiment and has been standardized to -1 and 1. 
            * Positive sentiment, if compound >= 0.05
            * Negative sentiment, if compound <=-0.05
            * Neutral sentiment, if -0.05 < compound < 0.05
    * VADER takes into account punctuation marks, acronyms and emoticons.
    * VADER gives a continuous score (-1 to 1), but in machine learning, we often convert it into a `categories (0/1 or 0/1/2)`. 
    * It should be noted that when converting into binary classification, there are two possible compound threshold values:
        * if compound >= 0:
                sentiment = 1
            else:
                sentment = 1
        * if compound >= 0.05:
                sentiment = 1
            else:
                sentiment = 0 
        
* Installation: in Jupyter Notebook, enter "!pip install vaderSentiment" 

## VADER analyzer

In [1]:
# -- load libraries --
import numpy as np
import pandas as pd

In [2]:
# -- load the class and instantiate an object --
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyser = SentimentIntensityAnalyzer()

## Uber_Rider_Review

In [3]:
# -- load the data --
df = pd.read_csv('https://raw.githubusercontent.com/ttchuang/dataset/refs/heads/master/Uber_Ride_Reviews.csv')
df.shape

(1344, 4)

In [4]:
# -- distribution of the sentiment --
df['sentiment'].value_counts()

sentiment
0    1111
1     233
Name: count, dtype: int64

In [5]:
# -- Define a printing sentiment score function --
def print_sentiment_scores(sentence):
    snt = analyser.polarity_scores(sentence)
    print("{:-<40}{}".format(sentence,str(snt)))

## Sample

In [6]:
# -- data sample --
df_sample = df.sample(n=100)
df_sample

,Key,ride_review,ride_rating,sentiment
836,837,Last nite got ride bad leg see hospital though...,1,0
267,268,Uber driver Zlata New Orleans charge cleaning ...,1,0
1210,1211,I use Uber When left concert July available ri...,1,0
624,625,I work Uber even add people company deer hit c...,1,0
475,476,Last night Uber driver Dietrich Beach took dau...,1,0
...,...,...,...,...
1129,1130,Used Uber first time last Sunday around Chicag...,1,0
681,682,The driver pick son others Orioles game colleg...,1,0
35,36,Uber rides always convenient plentiful prompt ...,5,1
432,433,My daughter left new boots Blundestones Uber c...,4,1


## Test

In [7]:
# -- preview columns --
df_sample.columns

Index(['Key', 'ride_review', 'ride_rating', 'sentiment'], dtype='object')

In [8]:
# -- preview sample --
df_sample.head()

,Key,ride_review,ride_rating,sentiment
836,837,Last nite got ride bad leg see hospital though...,1,0
267,268,Uber driver Zlata New Orleans charge cleaning ...,1,0
1210,1211,I use Uber When left concert July available ri...,1,0
624,625,I work Uber even add people company deer hit c...,1,0
475,476,Last night Uber driver Dietrich Beach took dau...,1,0


In [9]:
# -- distribution of the sentiment --
df_sample['sentiment'].value_counts()

sentiment
0    81
1    19
Name: count, dtype: int64

In [10]:
# -- analyze the sentiment scores of the ride_review of the first record --
analyser.polarity_scores(df_sample.iloc[0,1])

{'neg': 0.211, 'neu': 0.676, 'pos': 0.113, 'compound': -0.7985}

In [11]:
# -- analyze the sentiment scores of the ride_review of the second record --
analyser.polarity_scores(df_sample.iloc[1,1])

{'neg': 0.128, 'neu': 0.872, 'pos': 0.0, 'compound': -0.5719}

In [12]:
# -- analyze the sentiment scores of the ride_review of the third record --
analyser.polarity_scores(df_sample.iloc[2,1])

{'neg': 0.044, 'neu': 0.877, 'pos': 0.078, 'compound': 0.1945}

In [13]:
# -- create sentiment labels --
labels = {1: 'positive',0:'negative'}
df['sentiment_label']=df['sentiment'].map(labels)
df.head()

,Key,ride_review,ride_rating,sentiment,sentiment_label
0,1,I completed running New York Marathon requeste...,1,0,negative
1,2,My appointment time auto repairs required earl...,1,0,negative
2,3,Whether I using Uber ride service Uber Eats or...,1,0,negative
3,4,Why hard understand I trying retrieve Uber cab...,1,0,negative
4,5,I South Beach FL I staying major hotel ordered...,1,0,negative


## Prediction

In [14]:
# -- Define functions to convert sentiment from continuous to categogical --

def format_output(output_dict):
    polarity = "neutral"
    if(output_dict['compound']>= 0.05):
        polarity = "positive"
    elif(output_dict['compound']<= -0.05):
        polarity = "negative"
    return polarity

def predict_sentiment(text):
    output_dict =  analyser.polarity_scores(text)
    return format_output(output_dict)

In [15]:
# -- Run the predictions --
df["vader_prediction"] = df["ride_review"].apply(predict_sentiment)
df.head()

,Key,ride_review,ride_rating,sentiment,sentiment_label,vader_prediction
0,1,I completed running New York Marathon requeste...,1,0,negative,positive
1,2,My appointment time auto repairs required earl...,1,0,negative,positive
2,3,Whether I using Uber ride service Uber Eats or...,1,0,negative,negative
3,4,Why hard understand I trying retrieve Uber cab...,1,0,negative,positive
4,5,I South Beach FL I staying major hotel ordered...,1,0,negative,negative


In [16]:
# -- Distribuition of predictions --
df['vader_prediction'].value_counts()

vader_prediction
negative    692
positive    580
neutral      72
Name: count, dtype: int64

In [17]:
# -- Records with neutral sentiment --
df[df['vader_prediction']=='neutral']

,Key,ride_review,ride_rating,sentiment,sentiment_label,vader_prediction
19,20,We used Uber times week Orlando vacation inste...,4,1,positive,neutral
38,39,I ordered Uber Seattle I getting cruise ship n...,1,0,negative,neutral
140,141,Both Uber vehicles clean old uncomfortable Alt...,4,1,positive,neutral
149,150,I used Uber world always prompt convenient wel...,5,1,positive,neutral
172,173,I never knew going difficult task contact Uber...,1,0,negative,neutral
...,...,...,...,...,...,...
1270,1271,I thought good one night I took use Uber Milwa...,1,0,negative,neutral
1303,1304,Total ripoff four mile ride Halloween quoted a...,1,0,negative,neutral
1314,1315,Next door neighbor arrived late airport flight...,1,0,negative,neutral
1318,1319,Do get rating stars driver one guidelines,3,1,positive,neutral


In [18]:
# -- Records with positive or negative sentiment --
df[df['vader_prediction'] !='neutral']

,Key,ride_review,ride_rating,sentiment,sentiment_label,vader_prediction
0,1,I completed running New York Marathon requeste...,1,0,negative,positive
1,2,My appointment time auto repairs required earl...,1,0,negative,positive
2,3,Whether I using Uber ride service Uber Eats or...,1,0,negative,negative
3,4,Why hard understand I trying retrieve Uber cab...,1,0,negative,positive
4,5,I South Beach FL I staying major hotel ordered...,1,0,negative,negative
...,...,...,...,...,...,...
1339,1340,everyone knows uber fast couple drinks night w...,2,0,negative,positive
1340,1341,For service asks credit card number right bat ...,1,0,negative,positive
1341,1342,great service hiring drivers know way around t...,2,0,negative,positive
1342,1343,Uber several problems make poor experience con...,1,0,negative,negative


In [19]:
# -- Retrieve non neutral records for evaluating performance --
df_non_neutral = df[df['vader_prediction'] != 'neutral']
df_non_neutral

,Key,ride_review,ride_rating,sentiment,sentiment_label,vader_prediction
0,1,I completed running New York Marathon requeste...,1,0,negative,positive
1,2,My appointment time auto repairs required earl...,1,0,negative,positive
2,3,Whether I using Uber ride service Uber Eats or...,1,0,negative,negative
3,4,Why hard understand I trying retrieve Uber cab...,1,0,negative,positive
4,5,I South Beach FL I staying major hotel ordered...,1,0,negative,negative
...,...,...,...,...,...,...
1339,1340,everyone knows uber fast couple drinks night w...,2,0,negative,positive
1340,1341,For service asks credit card number right bat ...,1,0,negative,positive
1341,1342,great service hiring drivers know way around t...,2,0,negative,positive
1342,1343,Uber several problems make poor experience con...,1,0,negative,negative


## Prediction Performance

In [20]:
# -- prediction accuracy --

from sklearn.metrics import accuracy_score, classification_report
accuracy = accuracy_score(df_non_neutral['sentiment_label'],
                          df_non_neutral['vader_prediction'])
print(f'Accuracy: {accuracy: .4f}')

Accuracy:  0.6470


In [21]:
print(classification_report(df_non_neutral['sentiment_label'],
                            df_non_neutral['vader_prediction']))

              precision    recall  f1-score   support

    negative       0.94      0.61      0.74      1057
    positive       0.30      0.80      0.44       215

    accuracy                           0.65      1272
   macro avg       0.62      0.71      0.59      1272
weighted avg       0.83      0.65      0.69      1272

